# 0. Install et import

In [4]:
#%pip install pandas
#%pip install pyarrow
import pyarrow as pa
import pandas as pd
import numpy as np
print("PyArrow version", pa.__version__)
print("Pandas version", pd.__version__)

PyArrow version 25.0.0
Pandas version 3.0.5


# 1. Chargement fichier et filtrage

In [ ]:
#Voir quelques lignes
from pyarrow.parquet import ParquetFile
pf= ParquetFile("../data/food.parquet")
first_ten_rows = next(pf.iter_batches(batch_size = 10)) 
df = pa.Table.from_batches([first_ten_rows]).to_pandas()
print(df)
#data = pd.read_parquet(path="../data/food.parquet", engine="pyarrow", columns=['col1'])
#print(data.head())
#data_optim = pa.Table.from_pandas(data)

   additives_n additives_tags allergens_tags   brands_tags   brands  \
0            0             []      [en:nuts]  [xx:Bovetti]  Bovetti   
1            0             []             []      [lagg-s]   Lagg's   
2            0             []             []      [lagg-s]   Lagg's   
3            0             []             []   [xx:lagg-s]   Lagg's   
4            0             []             []      [lagg-s]   Lagg's   
5            0             []             []      [lagg-s]   Lagg's   
6            0             []             []      [lagg-s]   Lagg's   
7            0             []             []      [lagg-s]   Lagg's   
8            0             []             []      [lagg-s]   Lagg's   
9            0             []             []      [lagg-s]   Lagg's   

                                          categories  \
0                                                NaN   
1                                               null   
2  Plant-based foods and beverages, Beverages, Ho.

In [ ]:
data_chosen = pd.read_parquet(path="../data/food.parquet", engine="pyarrow", columns=['product_name', 'brands', 'nutriments', 'nutriscore_score', 'countries_tags', 'ingredients'])
#print(data_chosen.head())
data_chosen_france= data_chosen[data_chosen['countries_tags'].astype(str).str.contains("france")]
nbr_products_sold_in_france= len(data_chosen[data_chosen['countries_tags'].astype(str).str.contains("france")])
#1.247.346 produits vendus en France

# 62.82% de nutriScore non renseignés, donc 37,18% ont un nutri score renseignés
data_chosen_france[['nutriscore_score']].isna().sum() / nbr_products_sold_in_france 

#le top 10 des marques représentés
data_chosen_france[['brands']].value_counts()[0:11]

energy_100g = sugars_100g = salt_100g = 0
for item_i in data_chosen_france["nutriments"]:
    try:
        for item_j in item_i:
            try :
                if item_j["100g"]:
                    if item_j["name"] == "energy":
                        energy_100g= energy_100g +1
                    elif item_j["name"] == "sugars":
                        sugars_100g= sugars_100g +1
                    elif item_j["name"] == "salt":
                        salt_100g= salt_100g +1
            except KeyError:
                break
    except TypeError:
        i=0
taux_energy_manquant = ((nbr_products_sold_in_france - energy_100g)/nbr_products_sold_in_france) * 100
taux_sugars_manquant = ((nbr_products_sold_in_france - sugars_100g)/nbr_products_sold_in_france) * 100
taux_salt_manquant = ((nbr_products_sold_in_france - salt_100g)/nbr_products_sold_in_france) * 100
print (f"Le taux de manquant en energie, en sucre et en sel est : {taux_energy_manquant:.2f}% ,  {taux_sugars_manquant:.2f}% , {taux_salt_manquant:.2f}% ")



Le taux de manquant en energie, en sucre et en sel est : 30.22% ,  38.00% , 42.45% 
